In [1]:
!pip install pubchempy
!pip install beautifulsoup4
!pip install requests
!pip install pandas

In [2]:
import requests
import pandas as pd
from io import StringIO
from bs4 import BeautifulSoup
import time

# 1. We must define a User-Agent so the website thinks we are a browser
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

url = "https://cb.imsc.res.in/imppat/phytochemical/Oroxylum indicum"

try:
    # 2. Get the page with headers
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Check if the request was successful

    soup = BeautifulSoup(response.content, 'html.parser')

    # 3. Find the table
    table = soup.find('table')

    if table:
        # 4. Use StringIO to satisfy the new pandas requirements
        df = pd.read_html(StringIO(str(table)))[0]
        print("Successfully scraped the table!")
        print(df.head())

        # Save to file
        df.to_csv("oroxylum_compounds.csv", index=False)
    else:
        print("Could not find a table on this page.")
        # Debug: Print the first 500 chars of the page to see what's happening
        print(soup.prettify()[:500])

except Exception as e:
    print(f"An error occurred: {e}")

Successfully scraped the table!
  Indian medicinal plant Plant part IMPPAT Phytochemical identifier  \
0       Oroxylum indicum       bark                     IMPHY003321   
1       Oroxylum indicum       bark                     IMPHY004155   
2       Oroxylum indicum       bark                     IMPHY004416   
3       Oroxylum indicum       bark                     IMPHY004877   
4       Oroxylum indicum       bark                     IMPHY005447   

                    Phytochemical name  \
0  Flavanone, 5,7-dihydroxy-6-methoxy-   
1                          Scutellarin   
2                           Oroxylin A   
3                              Nepetin   
4                    6-Hydroxyluteolin   

                                          References  
0                                 ISBN:9770972795006  
1                                 ISBN:9788171360536  
2  ISBN:9770972795006, ISBN:9780387706375, ISBN:9...  
3                                 ISBN:9770972795006  
4            

In [5]:
import pubchempy as pcp
import pandas as pd

# Load your scraped file
df = pd.read_csv("/content/oroxylum_compounds.csv")

def get_smiles(name):
    try:
        results = pcp.get_compounds(name, 'name')
        if results:
            return results[0].isomeric_smiles
    except Exception as e:
        return None
    return None

print("Fetching SMILES (this may take a few minutes)...")
df['SMILES'] = df['Phytochemical name'].apply(get_smiles)

# Drop any compounds we couldn't find
df = df.dropna(subset=['SMILES'])
df.to_csv("oroxylum_with_smiles.csv", index=False)
print(f"Success! Saved {len(df)} compounds with structural data.")

Fetching SMILES (this may take a few minutes)...


/tmp/ipykernel_10115/1212634659.py:11: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  return results[0].isomeric_smiles


Success! Saved 66 compounds with structural data.


In [6]:
df = pd.read_csv('/content/oroxylum_with_smiles.csv')
df

,Indian medicinal plant,Plant part,IMPPAT Phytochemical identifier,Phytochemical name,References,SMILES
0,Oroxylum indicum,bark,IMPHY003321,"Flavanone, 5,7-dihydroxy-6-methoxy-",ISBN:9770972795006,COC1=C(C2=C(C=C1O)O[C@@H](CC2=O)C3=CC=CC=C3)O
1,Oroxylum indicum,bark,IMPHY004155,Scutellarin,ISBN:9788171360536,C1=CC(=CC=C1C2=CC(=O)C3=C(C(=C(C=C3O2)O[C@H]4[...
2,Oroxylum indicum,bark,IMPHY004416,Oroxylin A,"ISBN:9770972795006, ISBN:9780387706375, ISBN:9...",COC1=C(C2=C(C=C1O)OC(=CC2=O)C3=CC=CC=C3)O
3,Oroxylum indicum,bark,IMPHY004877,Nepetin,ISBN:9770972795006,COC1=C(C2=C(C=C1O)OC(=CC2=O)C3=CC(=C(C=C3)O)O)O
4,Oroxylum indicum,bark,IMPHY005447,6-Hydroxyluteolin,ISBN:9770972795006,C1=CC(=C(C=C1C2=CC(=O)C3=C(O2)C=C(C(=C3O)O)O)O)O
...,...,...,...,...,...,...
61,Oroxylum indicum,NaN,IMPHY011797,Oleic acid,ISBN:9788172361792,CCCCCCCC/C=C\CCCCCCCC(=O)O
62,Oroxylum indicum,NaN,IMPHY011974,4-Hydroxycinnamic acid,ISBN:9788172361792,C1=CC(=CC=C1/C=C/C(=O)O)O
63,Oroxylum indicum,NaN,IMPHY012747,Scutellarein,PMID:26330757,C1=CC(=CC=C1C2=CC(=O)C3=C(O2)C=C(C(=C3O)O)O)O
64,Oroxylum indicum,NaN,IMPHY014836,beta-Sitosterol,ISBN:9780387706375,CC[C@H](CC[C@@H](C)[C@H]1CC[C@@H]2[C@@]1(CC[C@...


In [3]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 41.4 MB/s eta 0:00:00


In [10]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def is_drug_like(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: return False

    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)

    # Rule of 5
    return mw < 500 and logp < 5 and hbd < 5 and hba < 10

df['is_valid'] = df['SMILES'].apply(is_drug_like)
final_list = df[df['is_valid'] == True]

final_list.to_csv("oroxylum_gnn_ready.csv", index=False)
print(f"Filtered list saved: {len(final_list)} compounds ready for GNN screening.")

Filtered list saved: 39 compounds ready for GNN screening.


In [11]:
df = pd.read_csv('/content/oroxylum_gnn_ready.csv')
df

,Indian medicinal plant,Plant part,IMPPAT Phytochemical identifier,Phytochemical name,References,SMILES,is_valid
0,Oroxylum indicum,bark,IMPHY003321,"Flavanone, 5,7-dihydroxy-6-methoxy-",ISBN:9770972795006,COC1=C(C2=C(C=C1O)O[C@@H](CC2=O)C3=CC=CC=C3)O,True
1,Oroxylum indicum,bark,IMPHY004416,Oroxylin A,"ISBN:9770972795006, ISBN:9780387706375, ISBN:9...",COC1=C(C2=C(C=C1O)OC(=CC2=O)C3=CC=CC=C3)O,True
2,Oroxylum indicum,bark,IMPHY004877,Nepetin,ISBN:9770972795006,COC1=C(C2=C(C=C1O)OC(=CC2=O)C3=CC(=C(C=C3)O)O)O,True
3,Oroxylum indicum,bark,IMPHY005513,Chrysin,"ISBN:9770972795006, ISBN:9780387706375, ISBN:9...",C1=CC=C(C=C1)C2=CC(=O)C3=C(C=C(C=C3O2)O)O,True
4,Oroxylum indicum,bark,IMPHY005607,Baicalein,"ISBN:9770972795006, ISBN:9780387706375, ISBN:9...",C1=CC=C(C=C1)C2=CC(=O)C3=C(O2)C=C(C(=C3O)O)O,True
5,Oroxylum indicum,bark,IMPHY007483,Dihydrobaicalein,"ISBN:9780387706375, ISBN:9788171360536",C1C(OC2=C(C1=O)C(=C(C(=C2)O)O)O)C3=CC=CC=C3,True
6,Oroxylum indicum,bark,IMPHY011505,"Apigenin 7,4'-dimethyl ether",ISBN:9770972795006,COC1=CC=C(C=C1)C2=CC(=O)C3=C(C=C(C=C3O2)OC)O,True
7,Oroxylum indicum,bark,IMPHY011974,4-Hydroxycinnamic acid,"ISBN:9770972795006, ISBN:9788171360536, ISBN:9...",C1=CC(=CC=C1/C=C/C(=O)O)O,True
8,Oroxylum indicum,bark,IMPHY012747,Scutellarein,ISBN:9788185042084,C1=CC(=CC=C1C2=CC(=O)C3=C(O2)C=C(C(=C3O)O)O)O,True
9,Oroxylum indicum,leaf,IMPHY000333,Aloe emodin,"ISBN:9770972795006, ISBN:9780387706375, ISBN:9...",C1=CC2=C(C(=C1)O)C(=O)C3=C(C2=O)C=C(C=C3O)CO,True
